# Task-0 Text Preprocessing

MVP preprocessing for IMDB and AG News before the Week 6 modeling tasks.

## Setup

In [1]:
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset

/home/atanas-vitanov/Documents/praksa/ml-interns-mk-2026/interns/atanas/week_6/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
from src.config import DATA_DIR, METADATA_DIR, PREPROCESSED_DATA_DIR, SEED
from src.preprocessing_utils import preprocess_dataset_splits
from src.result_saver import save_json

In [4]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [5]:
DATASET_CONFIGS = {
    "IMDB": {"hf_name": "stanfordnlp/imdb", "output_name": "imdb", "splits": ["train", "test"]},
    "AG News": {"hf_name": "fancyzhx/ag_news", "output_name": "ag_news", "splits": ["train", "test"]},
}

In [6]:
for output_dir in [METADATA_DIR, PREPROCESSED_DATA_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)

## Load Datasets

In [7]:
datasets_by_name = {}

for display_name, dataset_config in DATASET_CONFIGS.items():
    loaded_dataset = load_dataset(
        dataset_config["hf_name"],
        cache_dir=str(DATA_DIR / "huggingface"),
    )
    datasets_by_name[display_name] = {
        split_name: loaded_dataset[split_name]
        for split_name in dataset_config["splits"]
    }

In [8]:
datasets_by_name

{'IMDB': {'train': Dataset({
      features: ['text', 'label'],
      num_rows: 25000
  }),
  'test': Dataset({
      features: ['text', 'label'],
      num_rows: 25000
  })},
 'AG News': {'train': Dataset({
      features: ['text', 'label'],
      num_rows: 120000
  }),
  'test': Dataset({
      features: ['text', 'label'],
      num_rows: 7600
  })}}

## Preprocess

Steps: clean text, drop blanks, drop conflicting-label duplicates, deduplicate within splits, remove test rows that overlap with train, and save cleaned splits to `data/preprocessed/`.

In [9]:
preprocessing_results = {}

for display_name, dataset in datasets_by_name.items():
    preprocessing_results[display_name] = preprocess_dataset_splits(
        dataset=dataset,
        dataset_name=display_name,
        output_path=PREPROCESSED_DATA_DIR / DATASET_CONFIGS[display_name]["output_name"],
        text_column="text",
        label_column="label",
    )

Saving the dataset (1/1 shards): 100%|███████████████████████████████████████████████████████████████████████| 7588/7588 [00:00<00:00, 588930.23 examples/s]


In [10]:
preprocessing_results

{'IMDB': {'dataset': 'IMDB',
  'saved_path': '/home/atanas-vitanov/Documents/praksa/ml-interns-mk-2026/interns/atanas/week_6/data/preprocessed/imdb',
  'raw_summary': {'dataset': 'IMDB',
   'row_count': 50000,
   'blank_text_count': 0,
   'html_tag_count': 29201,
   'html_entity_count': 7,
   'control_character_count': 2,
   'repeated_unique_text_count': 410,
   'rows_with_repeated_text_count': 832,
   'conflicting_label_group_count': 0,
   'split_overlap_group_count': 123,
   'split_counts': {'test': 25000, 'train': 25000}},
  'clean_summary': {'dataset': 'IMDB',
   'row_count': 49578,
   'blank_text_count': 0,
   'html_tag_count': 0,
   'html_entity_count': 0,
   'control_character_count': 0,
   'repeated_unique_text_count': 0,
   'rows_with_repeated_text_count': 0,
   'conflicting_label_group_count': 0,
   'split_overlap_group_count': 0,
   'split_counts': {'test': 24676, 'train': 24902},
   'saved_path': '/home/atanas-vitanov/Documents/praksa/ml-interns-mk-2026/interns/atanas/week_

In [11]:
raw_summary_rows = []
clean_summary_rows = []
removal_summary_rows = []
explainability_examples = {}

for display_name, result in preprocessing_results.items():
    raw_summary_rows.append(result["raw_summary"])

    clean_summary = dict(result["clean_summary"])
    clean_summary["label_distribution"] = result["label_distribution"]
    clean_summary_rows.append(clean_summary)

    removal_summary = dict(result["removal_summary"])
    removal_summary["dataset"] = display_name
    removal_summary_rows.append(removal_summary)

    explainability_examples[display_name] = result["examples"]

In [12]:
raw_summary_df = pd.DataFrame(raw_summary_rows)
clean_summary_df = pd.DataFrame(clean_summary_rows)
removal_summary_df = pd.DataFrame(removal_summary_rows)

In [13]:
raw_summary_df

,dataset,row_count,blank_text_count,html_tag_count,html_entity_count,control_character_count,repeated_unique_text_count,rows_with_repeated_text_count,conflicting_label_group_count,split_overlap_group_count,split_counts
0,IMDB,50000,0,29201,7,2,410,832,0,123,"{'test': 25000, 'train': 25000}"
1,AG News,127600,0,0,5545,0,173,346,34,12,"{'test': 7600, 'train': 120000}"


In [14]:
clean_summary_df

,dataset,row_count,blank_text_count,html_tag_count,html_entity_count,control_character_count,repeated_unique_text_count,rows_with_repeated_text_count,conflicting_label_group_count,split_overlap_group_count,split_counts,saved_path,label_distribution
0,IMDB,49578,0,0,0,0,0,0,0,0,"{'test': 24676, 'train': 24902}",/home/atanas-vitanov/Documents/praksa/ml-inter...,"{'train': {'0': 12431, '1': 12471}, 'test': {'..."
1,AG News,127393,0,0,0,0,0,0,0,0,"{'test': 7588, 'train': 119805}",/home/atanas-vitanov/Documents/praksa/ml-inter...,"{'train': {'0': 29980, '1': 29977, '2': 29938,..."


In [15]:
removal_summary_df

,blank_text_rows,conflicting_label_rows,duplicate_rows_within_splits,test_overlap_rows,total_removed_rows,dataset
0,0,0,299,123,422,IMDB
1,0,68,129,10,207,AG News


## Explainability

Each table shows up to two examples per dataset for a preprocessing decision. Empty tables mean that issue was not present.

In [16]:
blank_text_examples_df = pd.DataFrame(
    [
        {"dataset_name": dataset_name, **row}
        for dataset_name, examples in explainability_examples.items()
        for row in examples["blank_text"]
    ]
)

In [17]:
blank_text_examples_df

""


In [18]:
text_artifact_examples_df = pd.DataFrame(
    [
        {"dataset_name": dataset_name, **row}
        for dataset_name, examples in explainability_examples.items()
        for row in examples["text_artifacts"]
    ]
)

In [19]:
text_artifact_examples_df

,dataset_name,dataset,split,label,raw_text,cleaned_text
0,IMDB,IMDB,train,0,I rented I AM CURIOUS-YELLOW from my video sto...,I rented I AM CURIOUS-YELLOW from my video sto...
1,IMDB,IMDB,train,0,If only to avoid making this type of film in t...,If only to avoid making this type of film in t...
2,AG News,AG News,train,3,HP: The Adaptive Enterprise that can't adapt &...,HP: The Adaptive Enterprise that can't adapt O...
3,AG News,AG News,train,3,Buffy the Censor Slayer &lt;strong&gt;Letters&...,Buffy the Censor Slayer Letters Readers drive ...


In [20]:
conflicting_label_examples_df = pd.DataFrame(
    [
        {"dataset_name": dataset_name, **row}
        for dataset_name, examples in explainability_examples.items()
        for row in examples["conflicting_labels"]
    ]
)

In [21]:
conflicting_label_examples_df

,dataset_name,dataset,split,label,raw_text,cleaned_text
0,AG News,AG News,train,2,Government Spending Up Sharply Locally Federa...,Government Spending Up Sharply Locally Federal...
1,AG News,AG News,train,3,Government Spending Up Sharply Locally Federal...,Government Spending Up Sharply Locally Federal...


In [22]:
duplicate_removal_examples_df = pd.DataFrame(
    [
        {"dataset_name": dataset_name, **row}
        for dataset_name, examples in explainability_examples.items()
        for row in examples["duplicate_rows_within_splits"]
    ]
)

In [23]:
duplicate_removal_examples_df

,dataset_name,dataset,split,label,raw_text,cleaned_text
0,IMDB,IMDB,train,0,I am not so much like Love Sick as I image. Fi...,I am not so much like Love Sick as I image. Fi...
1,IMDB,IMDB,train,0,Holy freaking God all-freaking-mighty. This mo...,Holy freaking God all-freaking-mighty. This mo...
2,AG News,AG News,train,0,"Phelps, Thorpe Advance in 200 Freestyle ATHENS...","Phelps, Thorpe Advance in 200 Freestyle ATHENS..."
3,AG News,AG News,train,2,Housing Starts Rebound Sharply in July WASHING...,Housing Starts Rebound Sharply in July WASHING...


In [24]:
test_overlap_examples_df = pd.DataFrame(
    [
        {"dataset_name": dataset_name, **row}
        for dataset_name, examples in explainability_examples.items()
        for row in examples["test_overlaps"]
    ]
)

In [25]:
test_overlap_examples_df

,dataset_name,dataset,split,label,raw_text,cleaned_text
0,IMDB,IMDB,test,0,An Italian/American co-production co-starring ...,An Italian/American co-production co-starring ...
1,IMDB,IMDB,test,0,"Oh my... bad clothing, worse synth music and t...","Oh my... bad clothing, worse synth music and t..."
2,AG News,AG News,test,1,China Sighs in Relief as Yao Scores High BEIJI...,China Sighs in Relief as Yao Scores High BEIJI...
3,AG News,AG News,test,3,Life without numbers in a unique Amazon tribe ...,Life without numbers in a unique Amazon tribe ...


## Save Metadata

In [26]:
task0_metadata = {
    "task": "Task 0 - Text Preprocessing",
    "seed": SEED,
    "datasets": DATASET_CONFIGS,
    "subsampling": "No subsampling used. Full train and test splits were scanned.",
    "preprocessing_steps": [
        "load train and test splits with HuggingFace load_dataset",
        "clean HTML/entities/control characters and normalize whitespace",
        "drop blank cleaned texts",
        "drop same cleaned text with different labels",
        "drop duplicate cleaned texts inside each split, keeping first",
        "drop overlapping cleaned texts from test",
        "save cleaned splits with HuggingFace DatasetDict.save_to_disk",
    ],
    "raw_summaries": raw_summary_rows,
    "clean_summaries": clean_summary_rows,
    "removal_summaries": removal_summary_rows,
    "explainability_examples": explainability_examples,
}

In [27]:
save_json(
    data=task0_metadata,
    output_path=METADATA_DIR / "task0_text_preprocessing_summary.json",
)

In [28]:
saved_outputs = [
    METADATA_DIR / "task0_text_preprocessing_summary.json",
    PREPROCESSED_DATA_DIR / "imdb",
    PREPROCESSED_DATA_DIR / "ag_news",
]

In [29]:
saved_outputs

[PosixPath('/home/atanas-vitanov/Documents/praksa/ml-interns-mk-2026/interns/atanas/week_6/results/metadata/task0_text_preprocessing_summary.json'),
 PosixPath('/home/atanas-vitanov/Documents/praksa/ml-interns-mk-2026/interns/atanas/week_6/data/preprocessed/imdb'),
 PosixPath('/home/atanas-vitanov/Documents/praksa/ml-interns-mk-2026/interns/atanas/week_6/data/preprocessed/ag_news')]

## Final Conclusion

Both datasets are now cleaned and saved in `data/preprocessed/`.
There were no blank texts, but there were HTML/text artifacts, repeated texts, and some train/test overlaps.
AG News also had some same-text examples with different labels, so those rows were removed.
The cleaned splits should be safer to use for the next tasks because exact duplicates and test leakage were removed.